# Script de Limpieza - Eliminar Medallion Architecture

Este notebook elimina completamente la arquitectura Medallion implementada en Databricks, incluyendo tablas, esquemas y catálogos.

In [0]:
%python
dbutils.widgets.removeAll()

## 1. Inicializar Widgets

Se limpian y definen los parámetros de entrada necesarios para especificar el contenedor, almacenamiento y catálogo a limpiar.

In [0]:
%python
dbutils.widgets.text("nameContainer","unit-catalog")
dbutils.widgets.text("nameStorage","adlsmartdata0912")
dbutils.widgets.text("catalogo","catalog_ut_smartdata")

## 2. Configurar Rutas de Almacenamiento

Se recuperan los valores de los widgets y se construye la ruta completa del almacenamiento Azure Data Lake.

In [0]:
%python
# Obtener los valores de los widgets
nameContainer = dbutils.widgets.get("nameContainer")
nameStorage = dbutils.widgets.get("nameStorage")
catalogo = dbutils.widgets.get("catalogo")

ruta = f"abfss://{nameContainer}@{nameStorage}.dfs.core.windows.net"
print(f"Ruta Storage: {ruta}")

Ruta Storage: abfss://unit-catalog@adlsmartdata0912.dfs.core.windows.net


## 3. Eliminar Arquitectura Medallion

Se eliminan todas las tablas, esquemas y catálogos de la arquitectura Medallion (Bronze, Silver, Gold) de forma cascada.

In [0]:
%python
# ==============================
# Tablas por capa
# ==============================
bronze_tables = ["sensor_events"]
silver_tables = ["sensor_events_clean"]
gold_tables = ["rig_daily_summary"]

# ==============================
# Función para eliminar tablas dinámicamente
# ==============================
def drop_tables(catalog, schema, tables):
    for table in tables:
        sql = f"DROP TABLE IF EXISTS {catalog}.{schema}.{table};"
        print(f"Ejecutando: {sql}")
        spark.sql(sql)

# ==============================
# Eliminar tablas por capa
# ==============================
print("=== Eliminando tablas Bronze ===")
drop_tables(catalogo, "bronze", bronze_tables)

print("=== Eliminando tablas Silver ===")
drop_tables(catalogo, "silver", silver_tables)

print("=== Eliminando tablas Gold ===")
drop_tables(catalogo, "golden", gold_tables)

# ==============================
# Eliminar esquemas
# ==============================
schemas = ["raw", "bronze", "silver", "golden", "exploratory"]
for schema in schemas:
    sql = f"DROP SCHEMA IF EXISTS {catalogo}.{schema} CASCADE;"
    print(f"Ejecutando: {sql}")
    spark.sql(sql)

# ==============================
# Eliminar catálogo
# ==============================
sql = f"DROP CATALOG IF EXISTS {catalogo} CASCADE;"
print(f"Ejecutando: {sql}")
spark.sql(sql)

=== Eliminando tablas Bronze ===
Ejecutando: DROP TABLE IF EXISTS catalog_ut_smartdata.bronze.sensor_events;
=== Eliminando tablas Silver ===
Ejecutando: DROP TABLE IF EXISTS catalog_ut_smartdata.silver.sensor_events_clean;
=== Eliminando tablas Gold ===
Ejecutando: DROP TABLE IF EXISTS catalog_ut_smartdata.golden.rig_daily_summary;
Ejecutando: DROP SCHEMA IF EXISTS catalog_ut_smartdata.raw CASCADE;
Ejecutando: DROP SCHEMA IF EXISTS catalog_ut_smartdata.bronze CASCADE;
Ejecutando: DROP SCHEMA IF EXISTS catalog_ut_smartdata.silver CASCADE;
Ejecutando: DROP SCHEMA IF EXISTS catalog_ut_smartdata.golden CASCADE;
Ejecutando: DROP SCHEMA IF EXISTS catalog_ut_smartdata.exploratory CASCADE;
Ejecutando: DROP CATALOG IF EXISTS catalog_ut_smartdata CASCADE;


DataFrame[]